# docuchat judged evaluation on a Kaggle GPU

Settings → Accelerator: **GPU T4 x2**, Internet: **On**. Then **Run All**.
Answers: Qwen3 8B. Judge: Qwen3 14B. Both run locally through llama.cpp, so no API keys are needed.
When it finishes, download **`/kaggle/working/eval_results.zip`** from the Output panel on the right.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q --branch feat/docuchat-evaluation https://github.com/ffalathel/rag-app.git /kaggle/working/rag-app
%cd /kaggle/working/rag-app
!git log --oneline -1

In [ ]:
# llama.cpp with CUDA: the prebuilt wheel if one matches, otherwise built from source (~15 min)
%env CMAKE_ARGS=-DGGML_CUDA=on
!pip install -q llama-cpp-python==0.3.35 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
!pip install -q -e ".[local-llm]"
import llama_cpp; print("llama.cpp GPU offload:", llama_cpp.llama_supports_gpu_offload())

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files

def q4(repo):
    name = next(f for f in list_repo_files(repo) if f.endswith("Q4_K_M.gguf"))
    return hf_hub_download(repo, name, local_dir="/kaggle/working/models")

ANSWER_GGUF = q4("Qwen/Qwen3-8B-GGUF")
JUDGE_GGUF = q4("Qwen/Qwen3-14B-GGUF")
print(ANSWER_GGUF, JUDGE_GGUF)

In [ ]:
import os
os.environ.update(
    DOCUCHAT_LLM_PROVIDER="llamacpp",
    DOCUCHAT_GGUF_PATH=ANSWER_GGUF,
    DOCUCHAT_JUDGE_MODEL=JUDGE_GGUF,
    DOCUCHAT_CONTEXT_WINDOW="8192",
)
# --workers 1: llama.cpp is not thread-safe
!python -m docuchat.evaluate run --judge --workers 1

In [ ]:
!cd /kaggle/working/rag-app && zip -qr /kaggle/working/eval_results.zip eval/results
print(open("eval/results/summary.md").read())